In [ ]:
###########################
#  BRANDS TINY-2
############################

import torch
from transformers import AutoTokenizer, AutoModel
import torch.nn.functional as F
import numpy as np
import pandas as pd
import re
import unicodedata


# -----------------------
# Модель и токенизатор
# -----------------------
MODEL_NAME = "cointegrated/rubert-tiny2"
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
base_model = AutoModel.from_pretrained(MODEL_NAME).to(DEVICE)

class VolumeClassifier(torch.nn.Module):
    def __init__(self, base_model_name):
        super().__init__()
        self.base_model = AutoModel.from_pretrained(base_model_name)
        hidden_size = self.base_model.config.hidden_size
        self.classifier = torch.nn.Linear(hidden_size, 3)  

    def forward(self, input_ids, attention_mask):
        outputs = self.base_model(input_ids=input_ids, attention_mask=attention_mask)
        last_hidden = outputs.last_hidden_state
        logits = self.classifier(last_hidden)
        return logits

tokenizer = AutoTokenizer.from_pretrained("brand_ner_model_4")
model = VolumeClassifier(MODEL_NAME).to(DEVICE)
model.load_state_dict(torch.load("brand_ner_model_4/brand_ner_model.pt", map_location=DEVICE))
model.eval()

# -----------------------
# Словарь для инференса
# -----------------------

df = pd.read_csv("data/train_brand.csv", sep=";")

def remove_accents_latin_only(text):
    def repl(ch):
        if 'A' <= ch <= 'Z' or 'a' <= ch <= 'z':
            decomp = unicodedata.normalize('NFD', ch)
            return ''.join(c for c in decomp if unicodedata.category(c) != 'Mn')
        return ch
    return ''.join(repl(ch) for ch in text)

def clean_text(text: str) -> str:
    if not isinstance(text, str):
        return ""

    # заменяем латинское ë на русское ё
    text = text.replace("ë", "ё").replace("Ë", "Ё")

    # убираем ударения только для латинских букв
    text = remove_accents_latin_only(text)

    # удаляем буквальные "\n" и "\t"
    text = re.sub(r"\\n|\\t", "", text)

    text = ''.join(
        ch for ch in text
        if unicodedata.category(ch)[0] != 'S' or ch == '№'
    )

    text = re.sub(r'\\u[0-9a-fA-F]{4}', '', text)

    # оставляем только нужные символы
    text = re.sub(r"[^a-zA-Zа-яА-ЯёЁ№!&0-9%\-–—,.'’_/ +]", "", text)

    # объединяем несколько пробелов
    text = re.sub(r"\s+", " ", text)

    return text.strip()

# Очистка столбца
df["sample"] = df["sample"].apply(clean_text)

inference_data = [ {"text": t} for t in df['sample']]

# -----------------------
# Функция инференса с агрегацией по словам
# -----------------------
def infer_volume_wordlevel(samples):
    results = []
    for item in samples:
        text = item["text"]
        encoding = tokenizer(
            text.split(),  # передаем уже split() слова
            is_split_into_words=True,
            return_tensors="pt",
            padding='max_length',
            truncation=True,
            max_length=30
        ).to(DEVICE)

        with torch.no_grad():
            logits = model(encoding["input_ids"], encoding["attention_mask"])
            probs = F.softmax(logits, dim=-1)  # [batch, seq_len, 3]

        # Привязка токенов к словам
        word_ids = encoding.word_ids(batch_index=0)
        current_word = None
        current_probs_1 = []
        current_probs_2 = []
        word_probs_1 = []
        word_probs_2 = []

        for w_id, p in zip(word_ids, probs.squeeze().cpu().numpy()):
            if w_id is None:
                continue

            if w_id != current_word:
                if current_word is not None:
                    # сохраняем максимальную вероятность по токенам слова
                    word_probs_1.append(float(np.max(current_probs_1)))
                    word_probs_2.append(float(np.max(current_probs_2)))
                current_word = w_id
                current_probs_1 = [p[1]]  # вероятность класса 1
                current_probs_2 = [p[2]]  # вероятность класса 2
            else:
                current_probs_1.append(p[1])
                current_probs_2.append(p[2])

        # для последнего слова
        if current_probs_1:
            word_probs_1.append(float(np.max(current_probs_1)))
            word_probs_2.append(float(np.max(current_probs_2)))

        results.append({
            "text": text,
            "brand_proba_1": word_probs_1,
            "brand_proba_2": word_probs_2,
        })

    return results



# -----------------------
# Запуск инференса
# -----------------------
output = infer_volume_wordlevel(inference_data)
df_out = pd.DataFrame(output)
df = pd.concat([df, df_out[['brand_proba_1', 'brand_proba_2']]], axis=1)
df

/tmp/ipykernel_21806/2018104145.py:38: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  model.load_state_dict(torch.load("brand_ner_model_4/brand_ner_model.pt", map_location=DE

In [ ]:
###########################
#  TYPES TINY-2
############################

import torch
from transformers import AutoTokenizer, AutoModel
import torch.nn.functional as F
import numpy as np
import pandas as pd
import torch.nn as nn

# -----------------------
# Модель и токенизатор
# -----------------------
MODEL_NAME = "cointegrated/rubert-tiny2"
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
base_model = AutoModel.from_pretrained(MODEL_NAME).to(DEVICE)

class VolumeClassifier(nn.Module):
    def __init__(self, base_model_name, num_labels=3):
        super().__init__()
        self.base = AutoModel.from_pretrained(base_model_name)
        hidden = self.base.config.hidden_size
        self.dropout = nn.Dropout(0.1)
        self.cls = nn.Linear(hidden, 3)

    def forward(self, input_ids, attention_mask):
        out = self.base(input_ids=input_ids, attention_mask=attention_mask)
        x = self.dropout(out.last_hidden_state)
        logits = self.cls(x)  # [B, T, C]
        return logits

tokenizer = AutoTokenizer.from_pretrained("type_ner_model_ilya")
model = VolumeClassifier(MODEL_NAME).to(DEVICE)
model.load_state_dict(torch.load("type_ner_model_ilya/type_ner_model.pt", map_location=DEVICE))
model.eval()

# -----------------------
# Функция инференса с агрегацией по словам
# -----------------------
def infer_volume_wordlevel(samples):
    results = []
    for item in samples:
        text = item["text"]
        encoding = tokenizer(
            text.split(),  # передаем уже split() слова
            is_split_into_words=True,
            return_tensors="pt",
            padding='max_length',
            truncation=True,
            max_length=30
        ).to(DEVICE)

        with torch.no_grad():
            logits = model(encoding["input_ids"], encoding["attention_mask"])
            probs = F.softmax(logits, dim=-1)  # [batch, seq_len, 3]

        # Привязка токенов к словам
        word_ids = encoding.word_ids(batch_index=0)
        current_word = None
        current_probs_1 = []
        current_probs_2 = []
        word_probs_1 = []
        word_probs_2 = []

        for w_id, p in zip(word_ids, probs.squeeze().cpu().numpy()):
            if w_id is None:
                continue

            if w_id != current_word:
                if current_word is not None:
                    # сохраняем максимальную вероятность по токенам слова
                    word_probs_1.append(float(np.max(current_probs_1)))
                    word_probs_2.append(float(np.max(current_probs_2)))
                current_word = w_id
                current_probs_1 = [p[1]]  # вероятность класса 1
                current_probs_2 = [p[2]]  # вероятность класса 2
            else:
                current_probs_1.append(p[1])
                current_probs_2.append(p[2])

        # для последнего слова
        if current_probs_1:
            word_probs_1.append(float(np.max(current_probs_1)))
            word_probs_2.append(float(np.max(current_probs_2)))

        results.append({
            "text": text,
            "type_proba_1": word_probs_1,
            "type_proba_2": word_probs_2,
        })

    return results



# -----------------------
# Запуск инференса
# -----------------------
output = infer_volume_wordlevel(inference_data)
df_out = pd.DataFrame(output)
df = pd.concat([df, df_out[['type_proba_1', 'type_proba_2']]], axis=1)
df

In [ ]:
# O

import torch
from transformers import AutoTokenizer, AutoModel
import torch.nn.functional as F
import numpy as np

# -----------------------
# Модель и токенизатор
# -----------------------
MODEL_NAME = "cointegrated/rubert-tiny2"
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
base_model = AutoModel.from_pretrained(MODEL_NAME).to(DEVICE)

# Если ты дообучал кастомный классификатор, загрузи его
class VolumeClassifier(torch.nn.Module):
    def __init__(self, base_model_name):
        super().__init__()
        self.base_model = AutoModel.from_pretrained(base_model_name)
        hidden_size = self.base_model.config.hidden_size
        self.classifier = torch.nn.Linear(hidden_size, 2)  # 0/1

    def forward(self, input_ids, attention_mask):
        outputs = self.base_model(input_ids=input_ids, attention_mask=attention_mask)
        last_hidden = outputs.last_hidden_state
        logits = self.classifier(last_hidden)
        return logits

tokenizer = AutoTokenizer.from_pretrained("o_ner_model")
model = VolumeClassifier(MODEL_NAME).to(DEVICE)
model.load_state_dict(torch.load("o_ner_model/o_ner_model.pt", map_location=DEVICE))
model.eval()

# -----------------------
# Функция инференса с агрегацией по словам
# -----------------------
def infer_volume_wordlevel(samples):
    results = []
    for item in samples:
        text = item["text"]
        encoding = tokenizer(
            text.split(),  # передаем уже split() слова
            is_split_into_words=True,
            return_tensors="pt",
            padding='max_length',
            truncation=True,
            max_length=30
        ).to(DEVICE)

        with torch.no_grad():
            logits = model(encoding["input_ids"], encoding["attention_mask"])
            probs = F.softmax(logits, dim=-1)
            probs_unit_tokens = probs[:, :, 1].squeeze().cpu().numpy()

        # Привязка токенов к словам
        word_ids = encoding.word_ids(batch_index=0)
        word_probs = []
        current_word = None
        current_probs = []
        for w_id, p in zip(word_ids, probs_unit_tokens):
            if w_id is None:
                continue
            if w_id != current_word:
                if current_word is not None:
                    word_probs.append(float(np.max(current_probs)))
                current_word = w_id
                current_probs = [p]
            else:
                current_probs.append(p)

        if current_probs:
            word_probs.append(float(np.max(current_probs)))

        results.append({
            "text": text,
            "o_probs": word_probs
        })
    return results


# -----------------------
# Запуск инференса
# -----------------------
output = infer_volume_wordlevel(inference_data)
df_out = pd.DataFrame(output)
df = pd.concat([df, df_out['o_probs']], axis=1)
df

In [ ]:
# PERCENT

import torch
from transformers import AutoTokenizer, AutoModel
import torch.nn.functional as F
import numpy as np

# -----------------------
# Модель и токенизатор
# -----------------------
MODEL_NAME = "cointegrated/rubert-tiny2"
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
base_model = AutoModel.from_pretrained(MODEL_NAME).to(DEVICE)

# Если ты дообучал кастомный классификатор, загрузи его
class VolumeClassifier(torch.nn.Module):
    def __init__(self, base_model_name):
        super().__init__()
        self.base_model = AutoModel.from_pretrained(base_model_name)
        hidden_size = self.base_model.config.hidden_size
        self.classifier = torch.nn.Linear(hidden_size, 3) 

    def forward(self, input_ids, attention_mask):
        outputs = self.base_model(input_ids=input_ids, attention_mask=attention_mask)
        last_hidden = outputs.last_hidden_state
        logits = self.classifier(last_hidden)
        return logits

tokenizer = AutoTokenizer.from_pretrained("percent_ner_model")
model = VolumeClassifier(MODEL_NAME).to(DEVICE)
model.load_state_dict(torch.load("percent_ner_model/percent_ner_model.pt", map_location=DEVICE))
model.eval()

# -----------------------
# Функция инференса с агрегацией по словам
# -----------------------
def infer_volume_wordlevel(samples):
    results = []
    for item in samples:
        text = item["text"]
        encoding = tokenizer(
            text.split(),  # передаем уже split() слова
            is_split_into_words=True,
            return_tensors="pt",
            padding='max_length',
            truncation=True,
            max_length=30
        ).to(DEVICE)

        with torch.no_grad():
            logits = model(encoding["input_ids"], encoding["attention_mask"])
            probs = F.softmax(logits, dim=-1)  # [batch, seq_len, 3]

        # Привязка токенов к словам
        word_ids = encoding.word_ids(batch_index=0)
        current_word = None
        current_probs_1 = []
        current_probs_2 = []
        word_probs_1 = []
        word_probs_2 = []

        for w_id, p in zip(word_ids, probs.squeeze().cpu().numpy()):
            if w_id is None:
                continue

            if w_id != current_word:
                if current_word is not None:
                    # сохраняем максимальную вероятность по токенам слова
                    word_probs_1.append(float(np.max(current_probs_1)))
                    word_probs_2.append(float(np.max(current_probs_2)))
                current_word = w_id
                current_probs_1 = [p[1]]  # вероятность класса 1
                current_probs_2 = [p[2]]  # вероятность класса 2
            else:
                current_probs_1.append(p[1])
                current_probs_2.append(p[2])

        # для последнего слова
        if current_probs_1:
            word_probs_1.append(float(np.max(current_probs_1)))
            word_probs_2.append(float(np.max(current_probs_2)))

        results.append({
            "text": text,
            "percent_proba_1": word_probs_1,
            "percent_proba_2": word_probs_2,
        })

    return results


# -----------------------
# Запуск инференса
# -----------------------
output = infer_volume_wordlevel(inference_data)
df_out = pd.DataFrame(output)
df = pd.concat([df, df_out[['percent_proba_1', 'percent_proba_2']]], axis=1)
df

In [ ]:
# VOLUME

import torch
from transformers import AutoTokenizer, AutoModel
import torch.nn.functional as F
import numpy as np

# -----------------------
# Модель и токенизатор
# -----------------------
MODEL_NAME = "cointegrated/rubert-tiny2"
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
base_model = AutoModel.from_pretrained(MODEL_NAME).to(DEVICE)

# Если ты дообучал кастомный классификатор, загрузи его
class VolumeClassifier(nn.Module):
    def __init__(self, base_model_name):
        super().__init__()
        self.base_model = AutoModel.from_pretrained(base_model_name)
        hidden_size = self.base_model.config.hidden_size
        self.classifier = nn.Linear(hidden_size, 3)

    def forward(self, input_ids, attention_mask):
        outputs = self.base_model(input_ids=input_ids, attention_mask=attention_mask)
        last_hidden = outputs.last_hidden_state
        logits = self.classifier(last_hidden)
        return logits
        
tokenizer = AutoTokenizer.from_pretrained("volume_ner_model")
model = VolumeClassifier(MODEL_NAME).to(DEVICE)
model.load_state_dict(torch.load("volume_ner_model/volume_ner_model.pt", map_location=DEVICE))
model.eval()

# -----------------------
# Функция инференса с агрегацией по словам
# -----------------------
def infer_volume_wordlevel(samples):
    results = []
    for item in samples:
        text = item["text"]
        encoding = tokenizer(
            text.split(),  # передаем уже split() слова
            is_split_into_words=True,
            return_tensors="pt",
            padding='max_length',
            truncation=True,
            max_length=30
        ).to(DEVICE)

        with torch.no_grad():
            logits = model(encoding["input_ids"], encoding["attention_mask"])
            probs = F.softmax(logits, dim=-1)  # [batch, seq_len, 3]

        # Привязка токенов к словам
        word_ids = encoding.word_ids(batch_index=0)
        current_word = None
        current_probs_1 = []
        current_probs_2 = []
        word_probs_1 = []
        word_probs_2 = []

        for w_id, p in zip(word_ids, probs.squeeze().cpu().numpy()):
            if w_id is None:
                continue

            if w_id != current_word:
                if current_word is not None:
                    # сохраняем максимальную вероятность по токенам слова
                    word_probs_1.append(float(np.max(current_probs_1)))
                    word_probs_2.append(float(np.max(current_probs_2)))
                current_word = w_id
                current_probs_1 = [p[1]]  # вероятность класса 1
                current_probs_2 = [p[2]]  # вероятность класса 2
            else:
                current_probs_1.append(p[1])
                current_probs_2.append(p[2])

        # для последнего слова
        if current_probs_1:
            word_probs_1.append(float(np.max(current_probs_1)))
            word_probs_2.append(float(np.max(current_probs_2)))

        results.append({
            "text": text,
            "volume_proba_1": word_probs_1,
            "volume_proba_2": word_probs_2,
        })

    return results


# -----------------------
# Запуск инференса
# -----------------------
output = infer_volume_wordlevel(inference_data)
df_out = pd.DataFrame(output)
df = pd.concat([df, df_out[['volume_proba_1', 'volume_proba_2']]], axis=1)
df

In [ ]:
df.to_csv("train_ensamble.csv", index=False)

In [ ]:
import numpy as np
import pandas as pd
import joblib
from ast import literal_eval
import re
import torch

# ======= Rule-based prior =======
VOLUME_RE  = re.compile(r"(?<!\w)(\d+[.,]?\d*)\s?(л|л\.|литр\w*|мл|ml|г|кг|шт|уп\w*|пак\w*|бут\w*)(?!\w)", re.IGNORECASE)
PERCENT_RE = re.compile(r"(?<!\w)(\d+[.,]?\d*)\s?%|процент\w*", re.IGNORECASE)

def rule_spans(text):
    out=[]
    for m in VOLUME_RE.finditer(text): out.append((m.start(), m.end(), "VOLUME"))
    for m in PERCENT_RE.finditer(text): out.append((m.start(), m.end(), "PERCENT"))
    return out

def apply_priors_to_probs(text, words, probs_row, beta=2.0):
    """
    words: list of слов
    probs_row: np.array, shape = (n_words, n_labels) с вероятностями моделей
    """
    offsets = []
    char_idx = 0
    for w in words:
        start = text.find(w, char_idx)
        end = start + len(w)
        offsets.append((start, end))
        char_idx = end + 1

    spans = rule_spans(text)
    for s, e, t in spans:
        for i, (w_start, w_end) in enumerate(offsets):
            if max(s, w_start) < min(e, w_end):  # пересечение
                if t == "VOLUME":
                    probs_row[i][4] += beta  # array(['BRAND', 'O', 'PERCENT', 'TYPE', 'VOLUME'], dtype='<U7')
                elif t == "PERCENT":
                    probs_row[i][2] += beta  # array(['BRAND', 'O', 'PERCENT', 'TYPE', 'VOLUME'], dtype='<U7')
    return probs_row

# --------------------------
# 1. Загружаем тестовый датасет
# --------------------------
test_data = pd.read_csv('all_labels_test.csv')
for col in test_data.columns:
    if col == 'sample':
        continue
    test_data[col] = test_data[col].apply(lambda x: literal_eval(x))

# --------------------------
# 2. Загружаем модели и LabelEncoder
# --------------------------
models = [joblib.load(f"xgb/xgb_fold{fold}.joblib") for fold in range(1, 6)]
le = joblib.load("xgb/label_encoder.joblib")

# --------------------------
# 3. Функция инференса с ансамблем
# --------------------------
def infer_annotations_ensemble(models, df, label_encoder):
    all_annotations = []

    for idx, row in df.iterrows():
        text = row['sample']
        words = text.split()
        n_words = len(words)

        # --------------------------
        # собираем фичи для каждого слова
        # --------------------------
        features = []
        for i, word in enumerate(words):
            # текущие вероятности
            probs = [
                row['brand_probs'][i],
                row['type_probs'][i],
                row['percent_probs'][i],
                row['volume_probs'][i],
                row['o_probs'][i]
            ]

            # позиция и длина
            word_pos = i
            word_len = len(word)

            # сосед слева
            if i > 0:
                prev_probs = [
                    row['brand_probs'][i-1],
                    row['type_probs'][i-1],
                    row['percent_probs'][i-1],
                    row['volume_probs'][i-1],
                    row['o_probs'][i-1]
                ]
                has_prev = 1
            else:
                prev_probs = [0, 0, 0, 0, 0]
                has_prev = 0

            # сосед справа
            if i < n_words - 1:
                next_probs = [
                    row['brand_probs'][i+1],
                    row['type_probs'][i+1],
                    row['percent_probs'][i+1],
                    row['volume_probs'][i+1],
                    row['o_probs'][i+1]
                ]
                has_next = 1
            else:
                next_probs = [0, 0, 0, 0, 0]
                has_next = 0

            feat = probs + [word_pos, word_len] + prev_probs + [has_prev] + next_probs + [has_next]
            features.append(feat)

        X_test = np.array(features, dtype=np.float32)

        # --------------------------
        # 3a. Получаем прогноз каждой модели
        # --------------------------
        all_probs = []
        for model in models:
            probs = model.predict_proba(X_test)  # shape = (n_words, n_classes)
            all_probs.append(probs)

        # --------------------------
        # 3b. Усредняем вероятности (soft voting)
        # --------------------------
        mean_probs = np.mean(all_probs, axis=0)
        mean_probs = apply_priors_to_probs(text, words, mean_probs, beta=1.0)
        y_pred_num = mean_probs.argmax(axis=1)
        y_pred_labels = label_encoder.inverse_transform(y_pred_num)

        # --------------------------
        # 3c. Формируем BIO-аннотации
        # --------------------------
        annotations = []
        prev_label = None
        char_idx = 0

        for i, word in enumerate(words):
            start_idx = text.find(word, char_idx)
            end_idx = start_idx + len(word)
            char_idx = end_idx + 1  # сдвигаем на 1 для пробела

            label = y_pred_labels[i]
            if label == 'O':
                bio_label = 'O'
                prev_label = None
            elif prev_label == label:
                bio_label = f'I-{label}'
            else:
                bio_label = f'B-{label}'

            prev_label = label
            annotations.append((start_idx, end_idx, bio_label))

        all_annotations.append(annotations)

    df['annotation'] = all_annotations
    return df

# --------------------------
# 4. Запускаем инференс
# --------------------------
test_data = infer_annotations_ensemble(models, test_data, le)

# --------------------------
# 5. Сохраняем результат
# --------------------------
test_data[['sample', 'annotation']].to_csv("xgboost_test_ensemble.csv", sep=";", index=False)
print("Inference complete. Saved to xgboost_test_ensemble.csv")